# show-o2-space
Run understanding, image generation, and selected-layer capture.
Select **Runtime > Change runtime type > GPU** first. Setup uses an isolated Python 3.11 environment; no runtime restart is required.
GPU inference is not yet verified. A100 is the initial target; T4 memory/precision support remains experimental.


In [ ]:
import subprocess
subprocess.run(["nvidia-smi"], check=True)


## 1. Clone project and original model implementation
Run setup once per fresh Colab runtime. This never deletes an existing checkout.


In [ ]:
from pathlib import Path
import os
import subprocess

def run(*args):
    subprocess.run(args, check=True)

PROJECT = Path("/content/show-o2-space")
UPSTREAM = Path("/content/Show-o")
if not PROJECT.exists():
    run("git", "clone", "https://github.com/Nicerova7/show-o2-space.git", str(PROJECT))
if not UPSTREAM.exists():
    run("git", "clone", "https://github.com/showlab/Show-o.git", str(UPSTREAM))
revision = "45a5a2de01d1ebd10cd5864d29310a76476cdf23"
run("git", "-C", str(UPSTREAM), "checkout", revision)
# Child Python processes import the original Show-o2 packages directly.
os.environ["PYTHONPATH"] = str(UPSTREAM / "show-o2") + os.pathsep + str(PROJECT)
os.chdir(PROJECT)


In [ ]:
# uv supplies a stable Python version independently of Colab's Python kernel.
import sys
run(sys.executable, "-m", "pip", "install", "uv==0.6.17")
run(sys.executable, "-m", "uv", "python", "install", "3.11")
ENV = Path("/content/show-o2-env")
if not ENV.exists():
    run(sys.executable, "-m", "uv", "venv", "--python", "3.11", str(ENV))
PYTHON = str(ENV / "bin/python")
run(sys.executable, "-m", "uv", "pip", "install", "--python", PYTHON,
    "torch==2.5.1", "torchvision==0.20.1", "--index-url", "https://download.pytorch.org/whl/cu121")
run(sys.executable, "-m", "uv", "pip", "install", "--python", PYTHON, "-r", "requirements.txt")
run(PYTHON, "-c", "import torch; from show_o2_space.adapter import ShowO2Adapter; print(torch.__version__, torch.cuda.get_device_name(0))")


## 2. Choose an image
Use the upstream example, or run the optional upload cell.


In [ ]:
import shutil
images = PROJECT / "inputs/images"
images.mkdir(parents=True, exist_ok=True)
source = UPSTREAM / "show-o2/docs/mmu/pexels-jane-pham-727419-1571673.jpg"
shutil.copy2(source, images / "example.jpg")
image_path = "inputs/images/example.jpg"
question = "Describe this image."
from IPython.display import display, Image
display(Image(filename=image_path, width=432))


In [ ]:
# Optional: upload a JPG or PNG from your computer.
from google.colab import files
uploaded = files.upload()
for name, data in uploaded.items():
    destination = images / Path(name).name
    destination.write_bytes(data)
    image_path = str(destination.relative_to(PROJECT))
if uploaded:
    question = "Describe this image."  # edit for your image
print(image_path, question)


## 3. Understanding and hidden states
Captures decoder blocks 0, 8, and 16 during prompt prefill.
Checks capture shapes and finite values; inspect the answer against the image.


In [ ]:
run(PYTHON, "-m", "show_o2_space.understanding", image_path, question, "--layers", "0", "8", "16")
print(Path("outputs/understanding/answer.txt").read_text())
run(PYTHON, "-c", """
import torch
from pathlib import Path
saved = torch.load('outputs/understanding/hidden_states.pt', map_location='cpu', weights_only=True)
states, metadata = saved['hidden_states'], saved['metadata']
alignment = metadata['alignment']
assert Path('outputs/understanding/answer.txt').read_text().strip(), 'Empty answer: inspect the run'
assert len(alignment['token_ids']) == alignment['sequence_length']
assert alignment['image_span'][1] - alignment['image_span'][0] == 729
assert set(states) == {0, 8, 16}
for hidden in states.values():
    assert hidden.shape == (1, alignment['sequence_length'], 1536)
    assert torch.isfinite(hidden).all(), 'Non-finite understanding activations'
print('Understanding output checks passed:', {k: tuple(v.shape) for k, v in states.items()})
""")


## 4. Text-to-image generation
The first call downloads weights; later commands reuse the download cache. Each command reloads the model.
Checks capture shapes and finite values plus image dimensions; inspect whether the image matches the prompt.


In [ ]:
prompt = "A photograph of a golden retriever sitting on green grass."
run(PYTHON, "-m", "show_o2_space.generation", prompt, "--seed", "42", "--layers", "0", "8", "16")
run(PYTHON, "-c", """
import torch
from PIL import Image
saved = torch.load('outputs/generation/hidden_states.pt', map_location='cpu', weights_only=True)
states = saved['hidden_states']
assert set(states) == {0, 8, 16}
for hidden in states.values():
    assert hidden.shape == (2, 1024, 1536)
    assert torch.isfinite(hidden).all(), 'Non-finite generation activations'
with Image.open('outputs/generation/image.png') as image:
    assert image.size == (432, 432) and image.mode == 'RGB'
    assert any(low != high for low, high in image.getextrema()), 'Constant generated image'
print('Generation output checks passed:', {k: tuple(v.shape) for k, v in states.items()})
""")
display(Image(filename="outputs/generation/image.png"))


## 5. Download outputs
Colab storage is temporary. Save the archive before disconnecting.


In [ ]:
from google.colab import files
archive = shutil.make_archive("/content/show-o2-space-outputs", "zip", PROJECT / "outputs")
files.download(archive)
